In [1]:
import glob 
import os
from openai import OpenAI

folders = glob.glob('myfiles')
folders

['myfiles']

In [3]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter

In [4]:
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.pdf",
                             loader_cls=PyPDFLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [5]:
len(documents)

166

In [6]:
documents[0]

Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': 'Lingrui Mei; Jiayu Yao; Yuyao Ge; Yiwei Wang; Baolong Bi; Yujun Cai; Jiazhi Liu; Mingyu Li; Zhong-Zhi Li; Duzhen Zhang; Chenlin Zhou; Jiayi Mao; Tianze Xia; Jiafeng Guo; Shenghua Liu', 'doi': 'https://doi.org/10.48550/arXiv.2507.13334', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'A Survey of Context Engineering for Large Language Models', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2507.13334v2', 'source': 'myfiles/2507.13334v2.pdf', 'total_pages': 166, 'page': 0, 'page_label': '1', 'doc_type': 'myfiles'}, page_content='A Survey of Context Engineering for Large\nLanguage Models\nLingrui Mei1,6,† Jiayu Yao1,6,† Yuyao Ge1,6,† Yiwei Wang2 Baolong Bi1,6,†\nYujun Cai3 Jiazhi Liu1 Mingyu Li1 Zhong-Zhi Li6 Duzhen Zhang6\nChenli

In [7]:
documents[0].metadata

{'producer': 'pikepdf 8.15.1',
 'creator': 'arXiv GenPDF (tex2pdf:)',
 'creationdate': '',
 'author': 'Lingrui Mei; Jiayu Yao; Yuyao Ge; Yiwei Wang; Baolong Bi; Yujun Cai; Jiazhi Liu; Mingyu Li; Zhong-Zhi Li; Duzhen Zhang; Chenlin Zhou; Jiayi Mao; Tianze Xia; Jiafeng Guo; Shenghua Liu',
 'doi': 'https://doi.org/10.48550/arXiv.2507.13334',
 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'title': 'A Survey of Context Engineering for Large Language Models',
 'trapped': '/False',
 'arxivid': 'https://arxiv.org/abs/2507.13334v2',
 'source': 'myfiles/2507.13334v2.pdf',
 'total_pages': 166,
 'page': 0,
 'page_label': '1',
 'doc_type': 'myfiles'}

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [10]:
!uv pip install -U -q langchain_chroma

In [11]:
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

In [12]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [13]:
db_name= "vector_db"

In [14]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name,
           embedding_function=embeddings).delete_collection()

In [15]:
vectorstore = Chroma.from_documents(documents=chunks,
                                    embedding=embeddings,
                                    persist_directory=db_name)

In [16]:
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 166 documents


In [17]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [18]:
result = vectorstore.similarity_search_with_relevance_scores("What is context engineering?", k=5)
print(result)

[(Document(id='b02cae7f-a621-4315-9552-f4bf44530d40', metadata={'creator': 'arXiv GenPDF (tex2pdf:)', 'page_label': '2', 'doc_type': 'myfiles', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'A Survey of Context Engineering for Large Language Models', 'creationdate': '', 'total_pages': 166, 'author': 'Lingrui Mei; Jiayu Yao; Yuyao Ge; Yiwei Wang; Baolong Bi; Yujun Cai; Jiazhi Liu; Mingyu Li; Zhong-Zhi Li; Duzhen Zhang; Chenlin Zhou; Jiayi Mao; Tianze Xia; Jiafeng Guo; Shenghua Liu', 'arxivid': 'https://arxiv.org/abs/2507.13334v2', 'doi': 'https://doi.org/10.48550/arXiv.2507.13334', 'page': 1, 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'producer': 'pikepdf 8.15.1', 'source': 'myfiles/2507.13334v2.pdf'}, page_content='Contents\n1 Introduction 4\n2 Related Work 5\n3 Why Context Engineering? 7\n3.1 Definition of Context Engineering . . . . . . . . . . . . . . . . . . . . . . .

In [19]:
vectorstore.similarity_search_with_score("what is the difference between prompt engineering and context engineering", k=3)

[(Document(id='e6cd6521-cab6-4167-b66a-0902edfc3a23', metadata={'doc_type': 'myfiles', 'total_pages': 166, 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'creator': 'arXiv GenPDF (tex2pdf:)', 'producer': 'pikepdf 8.15.1', 'author': 'Lingrui Mei; Jiayu Yao; Yuyao Ge; Yiwei Wang; Baolong Bi; Yujun Cai; Jiazhi Liu; Mingyu Li; Zhong-Zhi Li; Duzhen Zhang; Chenlin Zhou; Jiayi Mao; Tianze Xia; Jiafeng Guo; Shenghua Liu', 'page': 9, 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2507.13334v2', 'doi': 'https://doi.org/10.48550/arXiv.2507.13334', 'creationdate': '', 'source': 'myfiles/2507.13334v2.pdf', 'title': 'A Survey of Context Engineering for Large Language Models', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'page_label': '10'}, page_content='Dimension Prompt Engineering Context Engineering\nModel C=prompt (static string) C=A(c1,c2, . . . ,cn)(dynamic, structured assembly)\nTarget arg maxpromptPθ(Y

In [20]:
model='gpt-4o-mini'

# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=model)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

/tmp/ipykernel_34214/2674871582.py:7: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)


In [22]:
query = "Explain what is the difference between prompt engineering and context engineering"
result = conversation_chain.invoke({"question":query})
print(result["answer"])

Prompt engineering treats the context as a monolithic, static string of text, focusing on creating precise and contextually rich prompts to improve the performance of large language models (LLMs). In contrast, context engineering re-conceptualizes the context as a dynamically structured set of informational components that are sourced, filtered, and formatted through various functions. 

Key differences include:

1. **Complexity**: Prompt engineering is primarily manual and often focuses on specific tasks, while context engineering employs systematic optimization and can manage complexity through modular composition.

2. **Information**: In prompt engineering, the information content is fixed within the prompt. In context engineering, the aim is to maximize task-relevant information while managing the constraints of the context.

3. **State**: Prompt engineering is generally stateless, whereas context engineering is inherently stateful, incorporating explicit components for managing co

In [23]:
query = "exaplin in bullet points what is dynamic context assembly"
result = conversation_chain.invoke({"question":query})